**STEP 0: Install the library prerequisites**

---



In [ ]:
%pip3 install janus_swi ollama openai

**STEP 1.0: Define a system prompt and dynamic prompt generation function**

In [8]:
SYSTEM_PROMPT = """
You are an expert Prolog programmer. Given the context below, write a Prolog database that expresses the information in the context in the form of Prolog statements. Then, given the query below, write a Prolog query that can be used to answer the user's question. 
Reason your way through the problem step-by-step before providing the final Prolog code. Also capture implicit logical relationships between entities that might not be explicitly stated in the context. As such, you shall generate your own chain-of-thought.

Answer Format:
<reasoning step>
Your reasoning about generating a Prolog clause for the first piece of context.
<reasoning step>

...

<reasoning step>
Your reasoning about generating a Prolog clause for the last piece of context.
<reasoning step>


<reasoning step>
Your reasoning about the generating the Prolog query.
<reasoning step>

<prolog database>
Your final Prolog code that expresses the retrieved context in Prolog form.
<prolog database>

<prolog query>
Your final Prolog query that, when the Prolog engine is queried with, will answer the user's question.
<prolog query>

Example context-Prolog statement pairs:
Context: Bob is Alice's brother.
brother(alice, bob).

Context: Amanda is Alice's sister.
sister(alice, amanda).

Example database based on the example context:
<prolog database>
brother(alice, bob).
sister(alice, amanda).
<prolog database>

Example question-Prolog query pairs:
Question: Who is Alice's brother?
<prolog query>
brother(alice, X).
<prolog query>

Question: Who is Alice's sister?
<prolog query>
Query: sister(alice, X).
<prolog query>
"""

def generate_prompt(user_question: str, retrieved_context: str, most_recent_error: str) -> str:
    prompt = f"""
Context:
{retrieved_context}

User Question: 
{user_question}
"""

    if most_recent_error:
        prompt += f"""
Your recent Prolog database and query have errors. Fix them based on the following error message:
{most_recent_error}
"""

    return prompt

**STEP 1.1: Load the model directly using Ollama Library**

---



In [2]:
import ollama
ollama_messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
        ]
def generate_with_ollama(user_question, retrieved_context, most_recent_error) -> dict: 
    answer = dict()
    ollama_messages.append({'role': 'user', 'content': generate_prompt(user_question, retrieved_context, most_recent_error)})
    response = ollama.chat(
        model='llama3:instruct',
        messages=ollama_messages,
        options={
            'temperature': 0  
        },
        logprobs=True     
    )
    
    try: 
        answer['text_answer'] = response['message']['content']
        answer['logprobs'] = response.get('logprobs', None)
        return answer
    except Exception as e:
        print(f"Error during Ollama interaction: {e}")
        return None

**STEP 1.2: Load the model directly using OpenAI library Wrapper**

---



In [ ]:
from openai import OpenAI

openai_messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
        ]

# 1. Initialize the Client to hijack the local Ollama server
client = OpenAI(
            base_url='http://localhost:11434/v1',  # Points to your local machine
            api_key='ollama'                       # Required string, but ignored by Ollama
        )

def generate_with_openai(user_question, retrieved_context, most_recent_error) -> dict:
    answer = dict()
    openai_messages.append({'role': 'user', 'content': generate_prompt(user_question, retrieved_context, most_recent_error)})
    try:
        response = client.chat.completions.create(
            model="llama3:instruct",
            messages=openai_messages,
            temperature=0,      # 0 = Deterministic (Scientific standard)
            logprobs=True,      # Turn on confidence scores
            top_logprobs=5      # <--- THE KEY FEATURE: Get the top 5 alternatives per token
        )

        answer["text_answer"] = response.choices[0].message.content
        print(f"LLM Response:\n{answer['text_answer']}\n")

        # We inspect the first n tokens to see if the model hesitated.
        answer["logprobs"] = response.choices[0].logprobs.content
        return answer
    except Exception as e:
        print(f"Error during OpenAI client interaction: {e}")
        return None

**STEP 4: Create a function that uses regular expressions to capture the Prolog database and query in the LLM's response**

In [ ]:
def capture_db_and_query(answer_text: str) -> tuple[str, str]:
    import re

    db = ""
    query = ""
    return {"database":db, "query":query}

**STEP 2: Ask the LLM to generate two strings: the Prolog database and the query**

___

In [4]:
import janus_swi as janus

def generate_prolog_code(user_question: str, retrieved_context: str, most_recent_error: str):
    for i in range(5):
        try:
            answer = generate_with_openai(user_question, retrieved_context, most_recent_error)
            # database, query = capture_db_and_query(answer['text_answer'])["database"], capture_db_and_query(answer['text_answer'])["query"]
            # janus.consult("user", database)
            most_recent_error = None
            print(f"Iteration {i} | Prolog code generation success")
            return answer
        except Exception as e:
            print(f"Iteration {i} | Error during prompt generation or model interaction: {e}")
            continue
        except janus.PrologError as e:
            print(f"Iteration {i} | Prolog Error: {e}")
            most_recent_error = str(e)
            continue
        
    if most_recent_error:
        raise Exception("Failed to generate valid Prolog code after multiple attempts.")

In [9]:
user_question = "Based on the internal records, who is the uncle of Leo?"
retrieved_context = """
A recent internal audit of Apex Corp revealed the following organizational and personal ties: Mark serves as the Chief Executive Officer of the company and has one sibling, a brother named Steven. Separately, payroll records indicate that Steven is the primary guardian and father of a child named Leo.
"""
most_recent_error = None

code = generate_prolog_code(user_question, retrieved_context, most_recent_error)

Answer: <reasoning step>
To represent the context in Prolog form, we need to identify the key entities and relationships. We have Mark, Steven, Leo, and their organizational and personal ties.

<prolog database>

father(steven, leo).
brother(mark, steven).

<reasoning step>
The user's question is asking for the uncle of Leo. Since Steven is both the father and primary guardian of Leo, we can infer that he is also the uncle of Leo.

<prolog query>

uncle_of_leo(X) :-
    father(steven, leo),
    brother(mark, steven),
    X = steven.

This Prolog query will return `steven` as the answer when asked about the uncle of Leo.

Iteration 0 | Prolog code generation success
